In [1]:
# Cell 1 — Install all dependencies (~4 mins)
!pip install unsloth
!pip install gradio datasets transformers trl huggingface_hub
# Check GPU
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
# Expected: GPU: Tesla T4 | VRAM: 15.0 GB

GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# Cell 2 — Load Llama 3.2 with 4-bit quantization (QLoRA)
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
model_name = "unsloth/Llama-3.2-1B-Instruct",
max_seq_length = 2048,
dtype = None, # Auto-detect best dtype
load_in_4bit = True, # Saves 4x VRAM — must have this!
)
print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded successfully!


In [3]:
# Cell 3 — LoRA: only train 1% of parameters
model = FastLanguageModel.get_peft_model(
model,
r = 16, # Rank — 16 is sweet spot
target_modules = ["q_proj","k_proj","v_proj","o_proj",
"gate_proj","up_proj","down_proj"],
lora_alpha = 16,
lora_dropout = 0.05, # Small dropout prevents overfitting
bias = "none",
use_gradient_checkpointing = "unsloth",
random_state = 42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M of {total/1e6:.0f}M ({trainable/total*100:.1f}%)"
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.5 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable: 11.3M of 786M (1.4%)


In [4]:
import torch
from datasets import load_dataset
raw = load_dataset("lavita/MedQuAD", split="train")
print(f"Total examples: {len(raw)}") # ~47,000!
INSTRUCTION = "You are a helpful medical assistant. Answer clearly and safely."
PROMPT = """Below is an instruction with input. Write a helpful response.
### Instruction:
{instruction}
### Input:
{input}
### Response:
{output}"""
EOS = tokenizer.eos_token
def format_row(examples):
    texts = []
    for q, a in zip(examples["question"], examples["answer"]):
        if q and a: # skip empty rows
            texts.append(PROMPT.format(
                instruction=INSTRUCTION, input=q, output=a
            ) + EOS)
    return {"text": texts}
# Use first 2000 rows for faster training (still great quality)
dataset = raw.select(range(2000)).map(format_row, batched=True,
remove_columns=raw.column_names)
print(f"Training rows: {len(dataset)}")

Total examples: 47441
Training rows: 2000


In [5]:
# Cell 5 — Start training (~10-15 mins on free T4)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import gc # Import garbage collector

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 2, # 2 epochs — good balance
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = "outputs",
        save_strategy = "no", # Save after each epoch
        seed = 42,
        dataloader_num_workers = 0, # Fix: Set to 0 to avoid PicklingError with multiprocessing
    ),
)
print("Training started — check Loss values below...")
gc.collect() # Add garbage collection to potentially clean up object references
stats = trainer.train() # Assign the result of the first training call to stats
model.save_pretrained("my_model")
tokenizer.save_pretrained("my_model")
print(f"Done! Final loss: {stats.metrics['train_loss']:.4f}")
# Good loss = 1.0 to 1.8. If > 2.5 train for more epochs.

Training started — check Loss values below...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 2 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.170337
20,1.580053
30,1.317292
40,1.164832
50,1.178536
60,1.206925
70,1.070085
80,1.043356
90,1.168311
100,0.965553


Unsloth: Restored added_tokens_decoder metadata in my_model/tokenizer_config.json.


Done! Final loss: 1.0096


In [6]:
# Cell 6 — Save model locally in Colab
model.save_pretrained("medical_chatbot_lora")
tokenizer.save_pretrained("medical_chatbot_lora")
print("Saved locally!")

Unsloth: Restored added_tokens_decoder metadata in medical_chatbot_lora/tokenizer_config.json.


Saved locally!


In [7]:
# Cell 7 — Upload model and tokenizer to Hugging Face Hub

from google.colab import userdata
from huggingface_hub import login

# Token is read securely from Colab Secrets — never appears in this notebook
login(token=userdata.get('HF_TOKEN'))

# Push model and tokenizer separately (matches installed Unsloth version)
model.push_to_hub("dharaamehta33/medical-chatbot-llama")
tokenizer.push_to_hub("dharaamehta33/medical-chatbot-llama")

print("Uploaded! Visit: huggingface.co/dharaamehta33/medical-chatbot-llama")



Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.3kB / 45.1MB            

Saved model to https://huggingface.co/dharaamehta33/medical-chatbot-llama


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp9_jjwl9a/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp9_jjwl9a/tokenizer.json:  92%|#########2| 15.9MB / 17.2MB            

Uploaded! Visit: huggingface.co/dharaamehta33/medical-chatbot-llama
